In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"
import keras  # noqa: F401
print(keras.__version__)
print(keras.backend.backend())

3.14.1
torch


In [4]:
import torch
# 显示当前加速设备
if torch.cuda.is_available():
    print(f"当前加速设备: {torch.cuda.get_device_name(0)}")
elif torch.xpu.is_available():
    print(f"当前加速设备: {torch.xpu.get_device_name(0)}")
else:
    print("当前加速设备: CPU")

当前加速设备: Intel(R) Arc(TM) Graphics


In [7]:
(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# 直接对原始图像进行归一化
X_train_norm = X_train / 255.0
X_test_norm = X_test / 255.0

# 【关键步】：为卷积层增加“通道轴（Channel Axis）”
# 卷积层要求输入形状为 (样本数, 长, 宽, 通道数)。由于是灰度图，通道数是 1
X_train_conv = X_train_norm[:, :, :, None]
X_test_conv = X_test_norm[:, :, :, None]

print("训练集新形状:", X_train_conv.shape) # 应该显示 (60000, 28, 28, 1)
print("测试集新形状:", X_test_conv.shape) # 应该显示 (10000, 28, 28, 1)

训练集新形状: (60000, 28, 28, 1)
测试集新形状: (10000, 28, 28, 1)


In [8]:
@keras.saving.register_keras_serializable(package="CustomLayers")
class SwiGLULayer(keras.layers.Layer):
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        # 使用 Keras 3 标准的 self.add_weight
        self.W = self.add_weight(
            shape=(input_shape[-1], self.units), 
            initializer="glorot_uniform", 
            trainable=True, 
            name="W"
        )
        self.V = self.add_weight(
            shape=(input_shape[-1], self.units), 
            initializer="glorot_uniform", 
            trainable=True, 
            name="V"
        )
        self.b_W = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True, 
            name="b_W"
        )
        self.b_V = self.add_weight(
            shape=(self.units,), 
            initializer="zeros", 
            trainable=True, 
            name="b_V"
        )

    def call(self, inputs):
        # 使用 ops.matmul 代替 tf.matmul
        # 使用 ops.silu 代替 tf.nn.silu
        swish_branch = keras.ops.silu(keras.ops.matmul(inputs, self.W) + self.b_W)
        linear_branch = keras.ops.matmul(inputs, self.V) + self.b_V
        
        # 元素对应相乘（哈达玛积）
        return swish_branch * linear_branch

    def get_config(self):
        # 必须实现 get_config 以支持模型序列化
        config = super().get_config()
        config.update({"units": self.units})
        return config


In [9]:
# 1. 构建模型（使用 Keras 3 通用层）
model = keras.Sequential([

    keras.layers.Input(shape=(28, 28, 1)), 
    
    keras.layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.SpatialDropout2D(0.25),

    keras.layers.Conv2D(16, kernel_size=(3, 3), activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.SpatialDropout2D(0.25),
    
    keras.layers.MaxPooling2D(pool_size=(2, 2)),
    keras.layers.Flatten(),
    SwiGLULayer(128),
    keras.layers.GaussianDropout(0.3),
    keras.layers.Dense(10, activation='softmax')

])

In [10]:
# 模型编译
model.compile(optimizer="adam",
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# 模型训练并评估
model.fit(X_train_conv, y_train, epochs=20, batch_size=128, validation_data=(X_test_conv, y_test))

Epoch 1/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 29ms/step - accuracy: 0.7504 - loss: 0.7856 - val_accuracy: 0.8313 - val_loss: 0.4573
Epoch 2/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 30ms/step - accuracy: 0.8279 - loss: 0.4986 - val_accuracy: 0.8705 - val_loss: 0.3690
Epoch 3/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.8530 - loss: 0.4161 - val_accuracy: 0.8774 - val_loss: 0.3396
Epoch 4/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 14s 31ms/step - accuracy: 0.8669 - loss: 0.3733 - val_accuracy: 0.8882 - val_loss: 0.3114
Epoch 5/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - accuracy: 0.8805 - loss: 0.3278 - val_accuracy: 0.8984 - val_loss: 0.2812
Epoch 6/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 32ms/step - accuracy: 0.8904 - loss: 0.3015 - val_accuracy: 0.9018 - val_loss: 0.2710
Epoch 7/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 15s 31ms/step - accuracy: 0.8965 - loss: 0.2818 - val_accuracy: 0.9042 - val_loss: 0.2645
Epoch 8/20
469/469 ━━━━━━━━━━━━━━━━━━━━ 16s 34ms/step - accuracy: 0.9028 - loss: 0.2652 - 